In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest
import warnings

warnings.filterwarnings('ignore')

In [33]:
df = pd.read_csv("data.csv")

In [34]:
df.head(5)

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-05,(not set),mobile,Africa,Social Search,2,2,new account,1
1,2020-11-11,(not set),desktop,Asia,Direct,2,2,new account,1
2,2020-11-12,(not set),desktop,(not set),Direct,2,1,new account,1
3,2020-11-21,(not set),mobile,Americas,Organic Search,2,1,new account,1
4,2020-11-23,(not set),desktop,Africa,Organic Search,2,1,new account,1


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


In [36]:
df.describe()

,test,test_group,value
count,800996.000000,800996.000000,800996.000000
mean,2.831191,1.499608,9.475863
std,1.116940,0.500000,37.493267
min,1.000000,1.000000,1.000000
25%,2.000000,1.000000,1.000000
50%,3.000000,1.000000,2.000000
75%,4.000000,2.000000,5.000000
max,4.000000,2.000000,1575.000000


In [37]:
df['event_name'].value_counts()

event_name
session                107210
session_start          106242
page_view              101907
user_engagement         94520
first_visit             81621
scroll                  73643
view_promotion          61695
view_item               44869
session with orders     25892
new account             22389
view_search_results     14310
add_shipping_info       11961
begin_checkout          11959
select_item             11816
add_to_cart             11762
select_promotion         9044
add_payment_info         8048
click                    1997
view_item_list            111
Name: count, dtype: int64

In [38]:
df['date'] = pd.to_datetime(df['date'])

In [39]:
top_countries = df.groupby('country', as_index=False)['value'].sum().sort_values(by='value', ascending=False).reset_index(drop=True).head(5)['country']

In [40]:
top_countries

0     United States
1             India
2            Canada
3    United Kingdom
4            France
Name: country, dtype: object

In [41]:
segments = {'device': df['device'].unique(), 'continent': df['continent'].unique(), 'channel': df['channel'].unique(), 'country': top_countries}

metrics = df.groupby(['test', 'test_group', 'event_name'], as_index=False, dropna=False)['value'].agg('sum')

sessions = pd.Series([])
for i in metrics[['test', 'test_group']].values:
    sessions = pd.concat([sessions, metrics[((metrics['test'] == i[0]) & (metrics['event_name'] == 'session')) & ((metrics['test_group'] == i[1]) & (metrics['event_name'] == 'session'))]['value']])

metrics['sessions'] = sessions.reset_index(drop=True)
metrics['metric'] = metrics['value'] / metrics['sessions']
metrics['segment'] = 'Total'

for col, segment in segments.items():
    for seg in segment:
        seg_metrics = df[df[col] == seg].groupby(['test', 'test_group', 'event_name'], as_index=False, dropna=False)['value'].agg('sum')
                
        sessions = pd.Series([])
        for i in metrics[['test', 'test_group']].values:
            sessions = pd.concat([sessions, seg_metrics[((seg_metrics['test'] == i[0]) & (seg_metrics['event_name'] == 'session')) & ((seg_metrics['test_group'] == i[1]) & (seg_metrics['event_name'] == 'session'))]['value']])
                
        seg_metrics['sessions'] = sessions.reset_index(drop=True)
        seg_metrics['metric'] = seg_metrics['value'] / seg_metrics['sessions']
        seg_metrics['segment'] = seg

        metrics = pd.concat([metrics, seg_metrics])

In [42]:
metrics['segment'].unique()

array(['Total', 'mobile', 'desktop', 'tablet', 'Africa', 'Asia',
       '(not set)', 'Americas', 'Europe', 'Oceania', 'Social Search',
       'Direct', 'Organic Search', 'Paid Search', 'Undefined',
       'United States', 'India', 'Canada', 'United Kingdom', 'France'],
      dtype=object)

In [43]:
merged = pd.merge(
    metrics,
    metrics,
    how='inner',
    on=['test', 'event_name', 'segment']
)
 

In [44]:
merged = merged[(merged['test_group_x'] != merged['test_group_y'])].reset_index(drop=True)

In [45]:
merged = merged[merged['test_group_x'] != 2]

In [46]:
merged

,test,test_group_x,event_name,value_x,sessions_x,metric_x,segment,test_group_y,value_y,sessions_y,metric_y
0,1,1,add_payment_info,1988,45362,0.043825,Total,2,2229,45193,0.049322
1,1,1,add_shipping_info,3034,45362,0.066884,Total,2,3221,45193,0.071272
2,1,1,add_to_cart,1395,45362,0.030753,Total,2,1366,45193,0.030226
3,1,1,begin_checkout,3784,45362,0.083418,Total,2,4021,45193,0.088974
4,1,1,click,368,45362,0.008113,Total,2,353,45193,0.007811
...,...,...,...,...,...,...,...,...,...,...,...
2939,4,1,session_start,2109,2079,1.014430,France,2,2143,2107,1.017086
2940,4,1,user_engagement,5082,2079,2.444444,France,2,5200,2107,2.467964
2941,4,1,view_item,1938,2079,0.932179,France,2,1839,2107,0.872805
2942,4,1,view_promotion,1026,2079,0.493506,France,2,984,2107,0.467015


In [47]:
merged = merged.rename(columns={
    'test': 'test_number',
    'event_name': 'metric',
    'value_x': 'numerator_count_1',
    'sessions_x': 'denominator_count_1',
    'metric_x': 'conversion_rate_1',
    'value_y': 'numerator_count_2',
    'sessions_y': 'denominator_count_2',
    'metric_y': 'conversion_rate_2',    
})

In [48]:
merged = merged.drop(columns=['test_group_x', 'test_group_y'])

In [49]:
merged['numerator'] = merged['metric']
merged['denominator'] = 'session'

In [50]:
merged = merged[['segment', 'test_number', 'metric', 'numerator', 'denominator', 'numerator_count_1', 'denominator_count_1', 'conversion_rate_1', 'numerator_count_2', 'denominator_count_2', 'conversion_rate_2']]

In [51]:
merged['metric_change_%'] = 100 * (merged['conversion_rate_2']/merged['conversion_rate_1'] - 1)

In [52]:
merged[merged['metric'].isin(['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new account'])]

,segment,test_number,metric,numerator,denominator,numerator_count_1,denominator_count_1,conversion_rate_1,numerator_count_2,denominator_count_2,conversion_rate_2,metric_change_%
0,Total,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021
1,Total,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481
3,Total,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587
6,Total,1,new account,new account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299
38,Total,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911
...,...,...,...,...,...,...,...,...,...,...,...,...
2896,France,3,new account,new account,session,121,1475,0.082034,131,1444,0.090720,10.588700
2926,France,4,add_payment_info,add_payment_info,session,91,1444,0.063019,58,2079,0.027898,-55.731041
2927,France,4,add_shipping_info,add_shipping_info,session,119,2079,0.057239,78,2079,0.037518,-34.453782
2929,France,4,begin_checkout,begin_checkout,session,296,2079,0.142376,192,2107,0.091125,-35.997127


In [53]:
def z_test(data):
    return proportions_ztest([data['numerator_count_2'], data['numerator_count_1']], [data['denominator_count_2'], data['denominator_count_1']])

In [54]:
merged['z_stat'], merged['p_value'] = zip(*merged.apply(z_test, axis=1))

In [55]:
merged['significant'] = merged['p_value'] < 0.05

In [64]:
merged

,segment,test_number,metric,numerator,denominator,numerator_count_1,denominator_count_1,conversion_rate_1,numerator_count_2,denominator_count_2,conversion_rate_2,metric_change_%,z_stat,p_value,significant
0,Total,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,8.677153e-05,True
1,Total,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,9.225803e-03,True
2,Total,1,add_to_cart,add_to_cart,session,1395,45362,0.030753,1366,45193,0.030226,-1.712675,-0.460924,6.448531e-01,False
3,Total,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,2.893957e-03,True
4,Total,1,click,click,session,368,45362,0.008113,353,45193,0.007811,-3.717378,-0.510555,6.096629e-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2939,France,4,session_start,session_start,session,2109,2079,1.014430,2143,2107,1.017086,0.261811,NaN,NaN,False
2940,France,4,user_engagement,user_engagement,session,5082,2079,2.444444,5200,2107,2.467964,0.962161,NaN,NaN,False
2941,France,4,view_item,view_item,session,1938,2079,0.932179,1839,2107,0.872805,-6.369378,-6.468748,9.881798e-11,True
2942,France,4,view_promotion,view_promotion,session,1026,2079,0.493506,984,2107,0.467015,-5.368071,-1.715311,8.628828e-02,False


In [63]:
merged[(merged['segment'] == 'Total') & (merged['metric'].isin(['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new account']))]

,segment,test_number,metric,numerator,denominator,numerator_count_1,denominator_count_1,conversion_rate_1,numerator_count_2,denominator_count_2,conversion_rate_2,metric_change_%,z_stat,p_value,significant
0,Total,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,Total,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
3,Total,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
6,Total,1,new account,new account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
38,Total,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
39,Total,2,add_shipping_info,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
41,Total,2,begin_checkout,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
44,Total,2,new account,new account,session,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
76,Total,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
77,Total,3,add_shipping_info,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False


### Test 1
- add_payment_info - Статистично значуще покращення на 12.5%. 
- add_shipping_info - Статистично значуще покращення на 2.6%. 
- begin_checkout - Статистично значуще покращення на 3%. 
- new account - Статистично значущих змін немає.
##### Сonclusion
Тест показав що зміни в більшості метрик показали позитивні статистично значущі результати тому є сенс впровадити їх.
### Test 2
- add_payment_info - Статистично значущих змін немає. 
- add_shipping_info - Статистично значущих змін немає. 
- begin_checkout - Статистично значущих змін немає. 
- new account - Статистично значущих змін немає.
#### Conclusion
Тест показав що жодних статистично значущих змін не виявлено. Немає підстав впроваджувати.
### Test 3
- add_payment_info - Статистично значущих змін немає.
- add_shipping_info - Статистично значущих змін немає.
- begin_checkout - Статистично значуще погіршення на -3.3%. 
- new account - Статистично значущих змін немає.
#### Conclusion
Тест показав лише статистично значуще погіршення в метриці begin_checkout. Зміни є негативними тому впроваджувати їх не є доцільно.
### Test 4
- add_payment_info - Статистично значущих змін немає.
- add_shipping_info - Статистично значущих змін немає.
- begin_checkout - Статистично значуще погіршення на -2.3%. 
- new account - Статистично значуще погіршення на -3.3%. 
#### Conclusion
Тест показав статистично значуще погіршення в метриках begin_checkout та new account. Зміни є негативними тому впроваджувати їх не є доцільно.

In [58]:
merged.to_csv('dataframe.csv', header=True, index=False)

Tableau Dashboard: https://public.tableau.com/views/ABtestdashboard_17936323129630/ABtest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link